# Práctica 1 — Generación de texto y prompting con LLM

## Objetivo

Explorar la generación de texto con modelos de lenguaje mediante Hugging Face, comparando un modelo base con un modelo ajustado para seguir instrucciones.

Durante la práctica se analizaron:

- generación con GPT-2 en español;
- diferencias entre modelo base e instruction-tuned;
- efecto de `temperature`;
- efecto de `top_p`;
- efecto de `max_new_tokens`;
- uso de semillas para reproducibilidad;
- comparación de diferentes estrategias de prompting.

También se documentaron las limitaciones del entorno local y la selección de un modelo ligero compatible con el hardware disponible.

## Entorno de trabajo

- Python 3.12.5
- JupyterLab
- PyTorch en CPU
- Hugging Face Transformers
- Equipo con aproximadamente 8 GB de RAM
- GPU integrada AMD, sin CUDA

Debido a las limitaciones de memoria y al uso de CPU, no se priorizó la ejecución local completa de Gemma 2B IT. Su acceso mediante Hugging Face fue verificado, pero para los experimentos principales se utilizó un modelo más ligero.

In [1]:
import torch
import transformers 
import huggingface_hub

print("Pytorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("CUDA disponible:", torch.cuda.is_available())

Pytorch: 2.14.1+cpu
Transformers: 4.44.2
Hugging Face Hub: 0.25.2
CUDA disponible: False


In [2]:
from transformers import pipeline

modelo= "datificate/gpt2-small-spanish"

print("Modelo seleccionado:", modelo)
print("Preparado para iniciar la carga.")

Modelo seleccionado: datificate/gpt2-small-spanish
Preparado para iniciar la carga.


In [3]:
import torch
import transformers 
import huggingface_hub

print("Pytorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Huggin Face Hub:", huggingface_hub.__version__)
print("CUDA disponible:", torch.cuda.is_available())

Pytorch: 2.14.1+cpu
Transformers: 4.44.2
Huggin Face Hub: 0.25.2
CUDA disponible: False


In [4]:
import os

print("HF_HOME:", os.environ.get("HF_HOME"))

HF_HOME: D:\SEMILLERO_IA_UG\.cache\face


In [5]:
from transformers import pipeline

modelo = "datificate/gpt2-small-spanish"


In [6]:
modelo_chistes = pipeline(
    "text-generation",
    model= modelo,
    device= -1
)

D:\SEMILLERO_IA_UG\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [7]:
prompt = "Cuentame un chiste corto sobre inteligencia artificial."

resultado= modelo_chistes(
    prompt,
    max_new_tokens= 80,
    temperature= 0.9,
    top_p= 0.95
)[0]["generated_text"]

print("Prompt:")
print(prompt)

print("\nchiste generado:")
print(resultado)

Prompt:
Cuentame un chiste corto sobre inteligencia artificial.

chiste generado:
Cuentame un chiste corto sobre inteligencia artificial. De hecho, a la fecha esta canción está en el grupo sonora del programa "The X Factor".

Para la emisión de la segunda temporada, BBC One estrenó en el canal de Reino Unido las canciones "Bang to be my baby", "Heaven is my head" y "I'm a same".

El 12 de mayo de 2017 se anunció el lanzamiento de


In [8]:
print("do_sample por defecto:",
      modelo_chistes.model.generation_config.do_sample
     )
print("temperature por defecto:",
      modelo_chistes.model.generation_config.temperature
    )
print(
    "top_p por defecto:",
    modelo_chistes.model.generation_config.top_p
)

do_sample por defecto: True
temperature por defecto: 1.0
top_p por defecto: 1.0


In [9]:
resultado_a= modelo_chistes(
    prompt,
    max_new_tokens= 80,
    do_sample= False
)[0]["generated_text"]

print("=== A: SIN SAMPLING ===")
print(resultado_a)

=== A: SIN SAMPLING ===
Cuentame un chiste corto sobre inteligencia artificial.

En el año 2000, el programa de televisión "The X Factor" se estrenó en Estados Unidos. En el año 2002, el programa de televisión "The X Factor" se estrenó en Canadá. En el año 2003, el programa de televisión "The X Factor" se estrenó en Australia. En el año 2004, el programa de televisión "The X Factor" se estrenó en Canadá. En


In [10]:
resultado_b= modelo_chistes(
    prompt,
    max_new_tokens= 80,
    do_sample= True,
    temperature= 0.9,
    top_p= 0.95
)[0]["generated_text"]

print("=== B: CON SAMPLING ===")
print(resultado_b)

=== B: CON SAMPLING ===
Cuentame un chiste corto sobre inteligencia artificial. En las películas, la serie sigue a un profesor de matemáticas en una cafetería que ha sido secuestrada por extraterrestres, que ha perdido su trabajo en matemáticas y le ha dejado en estado de investigación.



Los "Tappings", la familia de ciencia ficción de The CW, han sido la serie más popular de los primeros tres programas en la década de 1990, seguido de la serie de


In [11]:
resultado_b= modelo_chistes(
    prompt,
    max_new_tokens= 80,
    do_sample= True,
    temperature= 0.9,
    top_p= 0.95
)[0]["generated_text"]

print("=== B: CON SAMPLING ===")
print(resultado_b)

=== B: CON SAMPLING ===
Cuentame un chiste corto sobre inteligencia artificial.

En 2018, se estrenó una película titulada "El día de hoy", producida por CBC y dirigida por Alan Seabor. La película narra el momento en que un grupo de exintegrantes de la banda de rock The Killers (Músicos del Rock) se reúnen para celebrar su aniversario número 100. El director de "El día de hoy" es Mark Wahlberg. La


In [ ]:
from getpass import getpass
from huggingface_hub import login

token= getpass("Token de Hugging Face: ")

login(
    token= token,
    add_to_git_credential= False
)

In [ ]:
from huggingface_hub import HfApi

api= HfApi()

usuario= api.whoami()

print("Usuario autenticado:", usuario["name"])

In [ ]:
from huggingface_hub import model_info

info_gemma= model_info("google/gemma-2b-it")

print("Modelo:", info_gemma.modelId)
print("Acceso confirmado")

In [ ]:
from huggingface_hub import hf_hub_download

ruta_config= hf_hub_download(
    repo_id= "google/gemma-2b-it",
    filename= "config.json"
)

print("Acceso a archivos de Gemma confirmado.")
print("Archivo:", ruta_config)

In [ ]:
del token


In [ ]:
from transformers import pipeline

modelo_instruct= "Qwen/Qwen2.5-0.5B-Instruct"

qwen= pipeline(
    "text-generation",
    model= modelo_instruct,
    device= -1
)

In [ ]:
mensajes= [
    {
        "role": "system",
        "content": "Eres un asistente util que responde en español."
    },
    {
        "role": "user",
        "content": "Cuentame un chiste corto sobre inteligencia artificial."
    }
]

In [ ]:
resultado_qwen= qwen(
    mensajes,
    max_new_tokens= 80,
    do_sample= False
)


In [ ]:
print(type(resultado_qwen))
print(resultado_qwen)

In [ ]:
respuesta_qwen= resultado_qwen[0]["generated_text"][-1]["content"]
print(respuesta_qwen)

resultado_qwen
    [0]
     ↓
primer resultado

["generated_text"]
     ↓
conversación generada

[-1]
     ↓
último mensaje

["content"]
     ↓
contenido del mensaje

## Comparación inicial de modelos

### datificate/gpt2-small-spanish
- Modelo base de generación de texto.
- Generó texto en español.
- No siguió correctamente la instrucción de contar un chiste.
- Produjo continuaciones sobre películas, novelas y otros temas.

### Qwen/Qwen2.5-0.5B-Instruct
- Modelo ajustado para seguir instrucciones.
- Comprendió que debía responder al usuario.
- Respondió en español.
- Permaneció relacionado con el tema solicitado.
- La calidad humorística todavía es limitada, pero el seguimiento de instrucciones es claramente superior.

### Conclusión
El ajuste para instrucciones afecta significativamente la capacidad de un modelo para interpretar lo que el usuario solicita. Los parámetros de generación no sustituyen esa capacidad.

In [ ]:
from transformers import set_seed

In [ ]:
def generar_con_temperatura(temperature):
    set_seed(42)

    resultado= qwen(
        mensajes,
        max_new_tokens= 80,
        do_sample= True,
        temperature= temperature,
        top_p= 0.95,
        top_k= 20
    )

    respuesta= resultado[0]["generated_text"][-1]["content"]

    return respuesta

In [ ]:
respuesta_t03= generar_con_temperatura(0.3)

print("=== TEMPERATURE 0.3 ===")
print(respuesta_t03)

In [ ]:
respuesta_t09= generar_con_temperatura(0.9)

print("=== TEMPERATURE 0.9 ===")
print(respuesta_t09)

In [ ]:
respuesta_t12= generar_con_temperatura(1.2)

print("=== TEMPERATURE 1.2 ===")
print(respuesta_t12)

## Experimento 1 — Efecto de temperature

Se mantuvieron constantes:
- Modelo: Qwen/Qwen2.5-0.5B-Instruct
- Prompt
- `do_sample=True`
- `top_p=0.95`
- `top_k=20`
- `max_new_tokens=80`
- semilla = 42

Resultados:

- `temperature=0.3`: respuesta corta, conservadora y coherente.
- `temperature=0.9`: respuesta más elaborada y diversa; mantiene inicialmente la coherencia, pero el remate final pierde sentido semántico.
- `temperature=1.2`: mayor variación, pero no mejor calidad.

### Conclusión

Aumentar `temperature` incrementa la diversidad potencial de la generación, pero una temperatura mayor no implica automáticamente una respuesta mejor. Existe un equilibrio entre variedad y coherencia.

In [ ]:
def generar_con_top_p(top_p):
    set_seed(42)

    resultado= qwen(
        mensajes,
        max_new_tokens= 80,
        do_sample= True,
        temperature= 0.9,
        top_p= top_p,
        top_k= 0
    )

    respuesta= resultado[0]["generated_text"][-1]["content"]

    return respuesta

In [ ]:
respuesta_p05= generar_con_top_p(0.5)

print("=== TOP_P 0.5 ===")
print(respuesta_p05)

In [ ]:
respuesta_p08= generar_con_top_p(0.8)
print("=== TOP_P 0.8 ===")
print(respuesta_p08)

In [ ]:
respuesta_p095= generar_con_top_p(0.95)

print("=== TOP_P 0.95 ===")
print(respuesta_p095)

## Experimento 2 — Efecto de top_p

Se mantuvieron constantes:

- Modelo: Qwen/Qwen2.5-0.5B-Instruct
- Prompt
- `do_sample=True`
- `temperature=0.9`
- `top_k=0`
- `max_new_tokens=80`
- semilla = 42

Resultados:

- `top_p=0.5`: respuesta breve y relativamente coherente, relacionada con inteligencia artificial.
- `top_p=0.8`: mayor variación, pero la respuesta se alejó del tema solicitado.
- `top_p=0.95`: mayor diversidad, aunque con una pérdida notable de coherencia semántica.

### Conclusión

Aumentar `top_p` amplía el conjunto de tokens candidatos durante el muestreo. Esto puede incrementar la diversidad de la generación, pero un conjunto más amplio no implica automáticamente una respuesta de mayor calidad.

Los resultados corresponden a este modelo, prompt, semilla y configuración concretos; no establecen que un valor de `top_p` sea universalmente mejor que otro.

In [ ]:
from transformers import set_seed
from transformers import pipeline

def generar_con_max_tokens(max_tokens):
    set_seed(42)

    resultado= qwen(
        mensajes,
        max_new_tokens= max_tokens,
        do_sample= True,
        temperature= 0.9,
        top_p= 0.8,
        top_k= 0
    )

    respuesta= resultado[0]["generated_text"][-1]["content"]

    return respuesta

In [ ]:
modelo_instruct= "Qwen/Qwen2.5-0.5B-Instruct"

qwen= pipeline(
    "text-generation",
    model= modelo_instruct,
    device= -1
)

mensajes= [
    {
        "role": "system",
        "content": "Eres un asistente util que responde en español"
        
    },
    {
        "role": "user",
        "content": "Cuentame un chiste corto sobre inteligencia artificial."
    }
]

In [ ]:
respuesta_m20= generar_con_max_tokens(20)

print("=== MAX_NEW_TOKENS 20 ===")
print(respuesta_m20)

In [ ]:
respuesta_m50= generar_con_max_tokens(50)

print("=== MAX_NEW_TOKENS 50 ===")
print(respuesta_m50)

In [ ]:
respuesta_m100= generar_con_max_tokens(100)

print("=== MAX_NEW_TOKENS 100 ===")
print(respuesta_m100)

## Experimento 3 — Efecto de max_new_tokens

Se mantuvieron constantes:

- Modelo: Qwen/Qwen2.5-0.5B-Instruct
- Prompt
- `do_sample=True`
- `temperature=0.9`
- `top_p=0.8`
- `top_k=0`
- semilla = 42

Resultados:

- `max_new_tokens=20`: la respuesta terminó de forma abrupta porque el límite fue insuficiente.
- `max_new_tokens=50`: el modelo pudo completar la respuesta.
- `max_new_tokens=100`: produjo la misma respuesta que con 50 tokens porque el modelo terminó naturalmente antes de alcanzar el máximo permitido.

### Conclusión

`max_new_tokens` establece un límite máximo de generación, no una cantidad obligatoria de tokens. Un valor demasiado bajo puede truncar la respuesta, mientras que aumentar el límite más allá de lo necesario no obliga al modelo a producir texto adicional.

In [ ]:
prompt_a = "Cuéntame un chiste corto sobre inteligencia artificial."

In [ ]:
prompt_b = """
Cuéntame un chiste corto sobre inteligencia artificial.
Debe estar en español, tener como máximo dos oraciones
y terminar con un remate claro.
"""

In [ ]:
prompt_c = """
Actúa como un escritor de humor tecnológico.

Escribe un único chiste breve sobre inteligencia artificial
para estudiantes universitarios de ingeniería de software.

Requisitos:
- Responde en español.
- Usa máximo dos oraciones.
- El tema central debe ser inteligencia artificial.
- Incluye un remate fácil de entender.
- No expliques el chiste.
"""

In [ ]:
prompt_d = """
Escribe un solo chiste breve en español sobre inteligencia artificial.

Usa formato de pregunta y respuesta.
La respuesta debe terminar con un remate claro.
Máximo 35 palabras.
No expliques el chiste.
"""

In [ ]:
def probar_prompt(prompt_usuario):
    set_seed(42)

    mensajes_prueba= [
        {
            "role": "system",
            "content": "Eres un asistente util que responde en español."
        },
        {
            "role": "user",
            "content": prompt_usuario
        }
    ]

    resultado = qwen(
        mensajes_prueba,
        max_new_tokens= 80,
        do_sample= True,
        temperature= 0.7,
        top_p= 0.8,
        top_k= 0
    )

    return resultado[0]["generated_text"][-1]["content"]

In [ ]:
respuesta_a= probar_prompt(prompt_a)

print("=== PROMPT A ===")
print(respuesta_a)

In [ ]:
respuesta_b= probar_prompt(prompt_b)

print("=== PROMPT B ===")
print(respuesta_b)

In [ ]:
respuesta_c= probar_prompt(prompt_c)

print("=== PROMPT C ===")
print(respuesta_c)

In [ ]:
respuesta_d= probar_prompt(prompt_d)

print("=== PROMPT D === ")
print(respuesta_d)

## Experimento 4 — Comparación de prompts

Se probaron cuatro niveles de prompting manteniendo constantes:

- Modelo: Qwen/Qwen2.5-0.5B-Instruct
- semilla = 42
- `temperature=0.7`
- `top_p=0.8`
- `top_k=0`
- `max_new_tokens=80`

### Resultados

- **Prompt A:** instrucción básica. Produjo la respuesta más sencilla y coherente como chiste.
- **Prompt B:** añadió restricciones, pero la respuesta perdió coherencia.
- **Prompt C:** incorporó rol, contexto y requisitos. Mantuvo mejor el tema de inteligencia artificial, pero no generó un remate satisfactorio.
- **Prompt D:** simplificó las restricciones del Prompt C, pero el remate continuó siendo poco coherente.

### Mejor prompt observado

El Prompt A produjo el mejor resultado en esta ejecución:

`Cuéntame un chiste corto sobre inteligencia artificial.`

Aunque era el prompt menos estructurado, permitió al modelo producir una respuesta más completa y lingüísticamente coherente.

### Conclusión

Un prompt más detallado no garantiza automáticamente una mejor salida. La efectividad depende de la tarea, del modelo y de su capacidad para seguir varias restricciones simultáneamente.

Estos resultados corresponden a una comparación controlada con una semilla y configuración determinadas y no implican que el Prompt A sea universalmente superior.

## Incidencias y aprendizajes

Durante la práctica se presentaron varias situaciones relevantes:

- Se evaluó el hardware antes de seleccionar el modelo local.
- Se verificó el acceso a Gemma 2B IT mediante Hugging Face, pero no se priorizó su ejecución local debido a las restricciones de RAM y CPU.
- GPT-2 en español permitió validar el pipeline, pero mostró limitaciones para seguir instrucciones.
- Se incorporó Qwen2.5-0.5B-Instruct como modelo instruction-tuned ligero para continuar los experimentos.
- Se diferenciaron errores de ejecución y warnings.
- Se comprobó que un reinicio del kernel elimina imports, variables y modelos cargados en memoria.
- Se utilizaron semillas para comparar configuraciones bajo condiciones reproducibles.
- Se comprobó que un prompt más complejo no garantiza automáticamente una mejor respuesta.

## Conclusiones

Esta práctica permitió comprender que la calidad de una generación no depende únicamente de modificar parámetros como `temperature`, `top_p` o `max_new_tokens`, sino también de las características y capacidades del modelo utilizado.

El modelo GPT-2 en español permitió validar el pipeline de generación, pero presentó limitaciones importantes para seguir instrucciones. La incorporación de Qwen2.5-0.5B-Instruct permitió trabajar con un modelo instruction-tuned más adecuado para analizar prompting y parámetros de generación dentro de las restricciones del hardware disponible.

Los experimentos mostraron que:

- `temperature` modifica el nivel de variabilidad de la generación, pero un valor mayor no implica mayor calidad.
- `top_p` controla el conjunto probabilístico de tokens candidatos y puede afectar diversidad y coherencia.
- `max_new_tokens` establece un límite máximo de salida y no una longitud obligatoria.
- una semilla fija ayuda a realizar comparaciones reproducibles.
- un prompt más elaborado no garantiza automáticamente una mejor respuesta.
- la selección de un modelo debe considerar tanto el objetivo de la tarea como los recursos disponibles.

La práctica también permitió aplicar criterios de selección de modelos, depuración de errores, manejo de Hugging Face, control de reproducibilidad y documentación de experimentos.